# 3.5 Tiling 源码走读：单模板八步逐段读

3.4 节看过了 Tiling 的执行框架和多模板调度。本节把镜头拉近，以 `FlashAttentionScoreTiling` 模板为例，把八步流程的代码逐段读懂。所有代码均来自 FA 算子真实实现（有删节）。

```text
① GetShapeAttrsInfo    获取 shape、attribute 信息
② GetPlatformInfo      获取硬件信息（核数/UB/L1/L0C）
③ IsCapable            判断是否支持当前场景
④ DoOpTiling           基本块选择、核间切分、负载均衡
⑤ DoLibApiTiling       核内 AscendC 高层 API tiling（Cube 侧）
⑥ GetWorkspaceSize     计算 workspace 大小
⑦ PostTiling           后处理（tilingData 大小、blockDim）
⑧ SetTilingKey         设置 TilingKey
```

## ① GetShapeAttrsInfo：从 context 解析输入

从框架传入的 context 中获取算子的输入、输出、属性，完成基本的合法性校验和简单的数据解析。

```cpp
ge::graphStatus FlashAttentionScoreTiling::GetShapeAttrsInfo()
{
    auto attrs = context_->GetAttrs();
    size_t idx = 0;
    // 按 IR 原型注册顺序取属性，索引递增
    auto scaleValuePtr = attrs->GetAttrPointer<float>(idx++);
    auto n1SizePtr = attrs->GetAttrPointer<uint32_t>(idx++);
    scaleValue = *scaleValuePtr;
    n1Size = *n1SizePtr;                       // Q 头数 N1

    auto &queryShape = context_->GetInputShape(0)->GetStorageShape();
    auto &keyShape   = context_->GetInputShape(1)->GetStorageShape();
    bSize  = queryShape.GetDim(0);
    s1Size = queryShape.GetDim(1);             // Q 的 S
    s2Size = keyShape.GetDim(1);               // K 的 S
    int64_t h1 = queryShape.GetDim(2);         // H = N1 × D
    int64_t h2 = keyShape.GetDim(2);
    dSize  = h1 / n1Size;                      // 单头维度 D
    gSize  = h1 / h2;                          // GQA 组数 G
    n2Size = h2 / dSize;                       // KV 头数 N2

    // 关键合法性校验（节选）
    CHECK_RET(h1 % n1Size != 0, ...);          // H 必须能整除出 D
    CHECK_RET(n1Size % n2Size != 0, ...);      // N1 必须是 N2 的倍数（GQA）
    CHECK_RET(dSize > HEAD_DIM_MAX_VALUE || dSize <= 0L, ...);

    // 按 FRACTAL_NUM（16）向上对齐
    alignedS1 = AlignUp(s1Size, FRACTAL_NUM);
    alignedS2 = AlignUp(s2Size, FRACTAL_NUM);
    alignedD  = AlignUp(dSize, FRACTAL_NUM);

    tilingData.inputParams.set_bSize(bSize);   // 填入 TilingData
    ...
    return ge::GRAPH_SUCCESS;
}
```

**看点**：BSH 类排布里第三维是 H 而非 N×D，GQA 的三个关键量 **G、N2、D 全部在这一步推导出来**（`dSize = h1/n1Size`，`gSize = h1/h2`，`n2Size = h2/dSize`）——3.3 节的排布知识在这里落地。

## ② GetPlatformInfo：硬件能提供什么

```cpp
ge::graphStatus FlashAttentionScoreTiling::GetPlatformInfo()
{
    auto platformInfoPtr = context_->GetPlatformInfo();
    auto ascendcPlatform = platform_ascendc::PlatformAscendC(platformInfoPtr);
    aivNum = ascendcPlatform.GetCoreNumAiv();   // AIV 核数
    aicNum = ascendcPlatform.GetCoreNumAic();   // AIC 核数
    ascendcPlatform.GetCoreMemSize(CoreMemType::UB,   aicoreParams_.ubSize);
    ascendcPlatform.GetCoreMemSize(CoreMemType::L1,   aicoreParams_.l1Size);
    ascendcPlatform.GetCoreMemSize(CoreMemType::L0_C, aicoreParams_.l0cSize);
    return ge::GRAPH_SUCCESS;
}
```

Tiling 的所有切分决策都以这些数字为边界——第 1 章讲的 UB / L1 / L0C 容量，就是在这里被读取使用的。

## ③ IsCapable：本模板管不管这个场景

```cpp
bool IsCapable()
{
    if (s2Size > s2sizeLimitMin) return true;
    return false;
}
```

每个模板只覆盖一部分输入范围（如 S2 下限），不匹配就返回 `GRAPH_PARAM_INVALID`，让框架去试下一个模板。

## ④ DoOpTiling：基本块 + 核间切分 + 负载均衡

这是 Tiling 的核心，三步走：

### MatchTemplate：选基本块

```cpp
bool FlashAttentionScoreTiling::MatchTemplate()
{
    // s1.i：默认 64；若核外 BNGS1 块数超过 AIV 核数，升到 128
    s1BasicBlock = std::min(64L, alignedS1);
    if (bSize * n1Size * gSize * CeilDiv(s1Size, s1BasicBlock) > aivNum) {
        s1BasicBlock = std::min(128L, alignedS1);
    }
    s2BasicBlock = std::min(128L, alignedS2);
    dBasicBlock  = std::min(128L, alignedD);
    // 高层 API 一次可用的 UB 预算
    apiMaxUBSize = HIGH_PERF_API_BUFFER_MULTIPLE
                   * s1BasicBlock * s2BasicBlock * sizeof(float);
    return true;
}
```

**看点**：`s1BasicBlock` 从 64 升到 128 的条件是「任务块数 > AIV 核数」——任务太多时切大块，减少调度开销；UB 预算公式 `s1s2·X + s1d·Y + s1·expNum·32 + s1·64 + apiTmp` 决定了基本块不能随意放大。

### SetCoreParams：核内参数

```cpp
void FlashAttentionScoreTiling::SetCoreParams()
{
    tilingData.coreParams.set_s1BaseSize(s1BasicBlock);
    tilingData.coreParams.set_s1OuterSize(CeilDivision(s1Size, s1BasicBlock));
    tilingData.coreParams.set_s2BaseSize(s2BasicBlock);
    tilingData.coreParams.set_nRatio(nRatio);   // Cube 配比
}
```

### SetMultiCoreParams：核间切分与负载均衡

```cpp
void FlashAttentionScoreTiling::SetMultiCoreParams()
{
    auto &multiCoreParams = tilingData.multiCoreParams;
    // 核间总任务数 = B × N2 × G × ceil(S1 / s1BasicBlock)
    int64_t totalSize = bSize * n2Size * gSize * tilingData.coreParams.get_s1OuterSize();
    actualUsedAivNum = std::min(totalSize, static_cast<int64_t>(aivNum));
    multiCoreParams.set_totalSize(totalSize);
    // 向上取整切分：以 AIV 为视角，确保最差的基本块性能最优
    multiCoreParams.set_splitFactorSize(CeilDivision(totalSize, actualUsedAivNum));
}
```

**看点**：

- 任务总数按 **B × N2 × G × S1 外层数** 计——正是 3.4 节「按 B、N2 分核」原则的代码形态；
- `splitFactorSize = ceil(total / actualAiv)`：把任务均分给实际用到的核，每个核领 `splitFactorSize` 个连续任务。

![Tiling 切分](images/fia_tiling_split.svg)

## ⑤ DoLibApiTiling：给 Cube 侧的高层 API 排产

FA 内部有 bmm1（Q·Kᵀ）和 bmm2（P·V）两次矩阵乘，以及一次 SoftMaxFlashV2。这些高层 API 自己也要排产（用多少 L1/L0C、单次算多大的 M/N/K）：

```cpp
bool FlashAttentionScoreTiling::SetBmm1TilingInput(...)
{
    bmm1.SetAType(TPosition::GM, CubeFormat::ND, DataType::DT_FLOAT16, false);
    bmm1.SetBType(TPosition::GM, CubeFormat::ND, DataType::DT_FLOAT16, true);   // B 转置
    bmm1.SetCType(TPosition::GM, CubeFormat::ND, DataType::DT_FLOAT);
    // 单次计算形状：M=s1 块，N=s2 块 × nRatio，K=D
    bmm1.SetShape(std::min(s1BasicBlock, s1Size),
                  std::min(s2BasicBlock * nRatio, s2Size), dSize);
    bmm1.SetOrgShape(s1Size, s2BasicBlock * nRatio, s1StrideSize, s2StrideSize);
    bmm1.SetBias(false);
    bmm1.SetBufferSpace(aicoreParams_.l1Size, aicoreParams_.l0cSize);   // 受 L1/L0C 约束
    ...
}
```

SoftMax 侧同理：

```cpp
void FlashAttentionScoreTiling::SetSoftMaxTiling()
{
    auto softmaxShape = ge::Shape({s1BasicBlock / nRatio, s2BasicBlock * nRatio});
    AscendC::SoftMaxFlashV2TilingFunc(softmaxShape, calcTypeSize, sizeof(float),
                                      apiMaxUBSize, tilingData.softmaxFlashTilingData,
                                      true, IsBasicBlockInSoftMax(softmaxShape));
}
```

**看点：nRatio 配比**。Vector 基本块是 `64 × 128`（s1 = 64），Cube 基本块配 `nRatio = 8` 份 Vector 块拼成 `64 × 1024` 的一次调用——**用 nRatio 把 Cube 与 Vector 的块大小配平**，减少两侧交互次数。

## ⑥ GetWorkspaceSize：算临时空间

```cpp
ge::graphStatus FlashAttentionScoreTiling::GetWorkspaceSize()
{
    size_t *workspaces = context_->GetWorkspaceSizes(1);
    int64_t bmm1Bytes = nRatio * s1BasicBlock * s2BasicBlock * calcTypeSize;

    if (dSize <= D_SPECIFIC_SIZE) {          // 无需切 D 的场景
        // stage1 占 2 倍空间，stage2 占 2 倍空间
        workspaces[0] = (bmm1Bytes * 2 + 2 * s1BaseSize * alignedD * calcTypeSize) * aivNum
                       + WORK_SPACE_RESERVE_SIZE;
        if (s2Size % 64 != 0) { /* s2 不对齐：stage1 升 3 倍 ... */ }
    } else {
        // 切 D 场景：stage1 2 倍，stage2 4 倍
        ...
    }
    return ge::GRAPH_SUCCESS;
}
```

workspace 包含：**CV 数据交换空间**（Cube 算完放哪、Vector 从哪取）、所有 Vector 核的计算空间、以及预留空间。倍数（2/3/4）取决于是否切 D、S2 是否对齐——尾块和不齐的维度会迫使流水线级数增加、占用翻倍。

## ⑦ PostTiling：收尾

```cpp
ge::graphStatus FlashAttentionScoreTiling::PostTiling()
{
    context_->GetRawTilingData()->SetDataSize(tilingData.GetDataSize());
    // CV 融合算子按「AIV+AIC 组合」启动：
    // 例如 40 AIV + 20 AIC，一个组合 = 2 AIV + 1 AIC，则 blockDim = 20
    auto blockDim = optiling::CalcTschBlockDim(actualUsedAivNum, aicNum, aivNum);
    context_->SetBlockDim(blockDim);
    return ge::GRAPH_SUCCESS;
}
```

**看点**：分离架构下 Kernel 不按「核数」启动，而按「**组合数**」启动——一个组合 = 若干 AIV + 1 AIC。`CalcTschBlockDim` 自动算出合适的 blockDim。这就是第 1 章组合核概念的接口落点。

## ⑧ SetTilingKey：盖上场景戳

最后按十进制位组装 TilingKey（最多支持 19 个参数），并 `DumpTilingInfo` 把 tilingKey 和 tilingData 记入日志。Kernel 侧将依据 TilingKey 在编译期就选中自己的代码段——呼应 3.4 节原则二。

![TilingKey 分发](images/tilingkey_dispatch.svg)

## 本节小结

| 步骤 | 产出 | 关键约束 |
|--|--|--|
| ① GetShapeAttrsInfo | B/S1/S2/D/G/N2 | GQA 维度全部推导于此 |
| ② GetPlatformInfo | aivNum/aicNum/UB/L1/L0C | 所有切分的物理边界 |
| ③ IsCapable | 支持 / 不支持 | 不支持 → 试下一个模板 |
| ④ DoOpTiling | 基本块、核间切分、splitFactorSize | UB 预算 + 任务数 vs 核数 |
| ⑤ DoLibApiTiling | bmm1/bmm2/softmax 排产 | nRatio 配比 Cube/Vector |
| ⑥ GetWorkspaceSize | workspace 字节数 | 尾块/不对齐 → 倍数翻升 |
| ⑦ PostTiling | tilingData 大小、blockDim | 按 AIV+AIC 组合启动 |
| ⑧ SetTilingKey | tilingKey | 编译期代码隔离 |

## 小测验

1. `queryShape` 第三维是 H = N1×D 时，Tiling 如何推导出 G、N2、D 三个量？
2. `s1BasicBlock` 什么条件下从 64 升到 128？背后的权衡是什么？
3. `nRatio` 在 bmm1 与 SoftMax 的 tiling 里各出现在哪里？它配平的是什么？
4. PostTiling 里 blockDim 为什么不等于 AIV 核数？以「40 AIV + 20 AIC、一个组合 2 AIV + 1 AIC」为例算一算。